# The coarse-correction step: what a finer level actually receives

The per-level panels (wandb `val/levels`, `vcycle_levels_real.ipynb`) show each level's
recon $w_\ell$. A finer level never uses that image. On the way up, `PDVCycle` adds

$$x_\ell \;\leftarrow\; x_\ell + \alpha_x\, P\big(w_{\ell+1} - u_{\ell+1}\big), \qquad u_{\ell+1} = R\,x_\ell ,$$

so only the coarse **change** crosses, and only **through $P$**. A coarse recon can be
badly wrong and still hand up a harmless correction if:
* its error sits in the band $P$ does not pass (the coarse grid's top band; $|H| = 1/2$
  at the coarse Nyquist and falls to 0 at the fine one), or
* the error was already in its starting point $u_{\ell+1}$, so it cancels in $w - u$, or
* $\alpha_x$ is small.

In each of those cases, nothing in the loss pushes the coarse level back to an image.

Levels are numbered as in the code and the wandb panel: **L0 = finest**, L2 = coarsest.
Each row below is one transition $L_{\ell+1} \to L_\ell$, coarsest first.

Engine: `scripts/level_figure.py` (`correction_parts`, `correction_table`,
`fig_correction`, `fig_correction_spectra`). The measurement is `eval_per_volume`'s for
the slice. Live mode only: it needs the checkpoint, the data, and torch for $P$.

In [ ]:
import pathlib, sys, os

_root = pathlib.Path.cwd()
if not (_root / "models").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "scripts"))
os.chdir(_root)                      # run dirs and the data roots are repo-relative

import numpy as np
import matplotlib.pyplot as plt
import importlib
import level_figure as lf
if getattr(lf, "_FILE_MTIME", None) != os.path.getmtime(lf.__file__):
    lf = importlib.reload(lf)
    lf._FILE_MTIME = os.path.getmtime(lf.__file__)

RUN     = "trained_nets/mg_recon/knee/mg81v6vtxA_R8"   # still has the missing-pi^1 FAS
VOLUME  = "1000108"         # any unique part of a knee val volume name
SLICE   = 22
SIGMA   = None              # None: the run's val_noise_std
VAL_KSPACE = "measurement_awgn"
OUTER   = -1                # which V-cycle (0-based; -1 = the last)
FLIP    = None              # None: 'ud' for knee, as the viewer

In [ ]:
z, meta, live = lf.trace_slice(RUN, VOLUME, SLICE, sigma=SIGMA, val_kspace_type=VAL_KSPACE)
FLIP = lf.default_flip(meta) if FLIP is None else FLIP
print(f"{meta['run']}  step {meta['step']}  {meta['n_params']:,} params   "
      f"{meta['volume']} slice {meta['slice']}  R={meta['R']}  sigma={meta['sigma']:g}")
print(f"transfer filter: {meta['transfer_filter']}   coarse op: {meta['coarse_op']}   "
      f"output NRMSE {100 * lf.nrmse(z['recon'], z['gt']):.2f}%")

## 1. The correction step, image domain

Columns:
1. $w_c$, the coarse recon.
2. $R\,\mathrm{gt}$ on the coarse grid.
3. The coarse error $|w_c - R\,\mathrm{gt}|$.
4. $P(w_c)$.
5. The part of the coarse error that $P$ lets through, $|P(w_c - R\,\mathrm{gt})|$.
6. What the coarse level hands up, $|P(w_c - u_c)|$.
7. What the fine level actually adds, $|\alpha_x P(\cdot)|$.
8. The fine level before the correction.
9. The fine level after it.

How the panels are scaled:
* Columns 3 and 5 share the error window (vmax / 5), so they compare directly.
* Columns 6 and 7 share their own window, printed in cyan, because a correction is much
  smaller than an error. Cyan also gives each one's RMS as a % of the fine image.
* Yellow is NRMSE. $P(w_c)$ is scored against $P(R\,\mathrm{gt})$, which isolates the coarse
  error that survives $P$.

The row label gives two numbers:
* **visible**: RMS of $P(\text{coarse error})$ / RMS of the coarse error. $P$ keeps the RMS of
  anything in its passband, so 100% means the whole error crosses.
* **alpha**: fitted from added = $\alpha \cdot P(w_c - u_c)$. The fit's misfit is in the table
  below; it is ~0 when the formula above holds.

In [ ]:
fig = lf.fig_correction(z, meta, outer=OUTER, flip=FLIP)
plt.show()

## 2. The same numbers for every V-cycle

* `u_c -> w_c`: NRMSE against $R\,\mathrm{gt}$ of where the coarse level **started**
  ($u_c = R\,x_{
m pre}$) and where it **ended**. If `w_c` is worse than `u_c`, the coarse
  layers moved away from the restricted fine iterate. If `u_c` is already poor, then "poor
  coarse" means the fine iterate, restricted, is poor.
* `x_pre -> x_corr`: the fine level just before and after it adds the correction. If
  `x_corr` is worse than `x_pre`, the correction hurt at that point. The post-smoother may
  still recover.
* `nrmse_w_c`: the coarse recon against $R\,\mathrm{gt}$, which is what the panels show.
* `nrmse_P_w`: $P(w_c)$ against $P(R\,\mathrm{gt})$.
* `visible`: as above.
* `step_rel`: RMS of what the fine level adds / RMS of the fine image.
* `alpha` and `fit misfit`: $\alpha_x$ recovered from the trace.

If the coarse recon degrades over V-cycles but `visible` × `nrmse_w_c` and `step_rel` stay
small, the fine levels are not being fed the corruption.

In [ ]:
rows = lf.correction_table(z, meta)
print(f"{'V':>2} {'transition':>10} {'u_c -> w_c':>16} {'x_pre -> x_corr':>16} "
      f"{'nrmse P(w_c)':>13} {'visible':>8} {'step_rel':>9} {'alpha':>8} {'fit misfit':>10}")
for r in rows:
    print(f"{r['outer']:>2} {r['transition']:>10} "
          f"{100 * r['nrmse_u_c']:>6.2f}% -> {100 * r['nrmse_w_c']:>5.2f}% "
          f"{100 * r['nrmse_x_pre']:>6.2f}% -> {100 * r['nrmse_x_corr']:>5.2f}% "
          f"{100 * r['nrmse_P_w']:>12.2f}% {100 * r['visible']:>7.1f}% "
          f"{100 * r['step_rel']:>8.3f}% {r['alpha'].real:>+8.3f} {r['alpha_misfit']:>10.1e}")

## 3. Frequency content: the coarse recon vs what crosses $P$

Each row is one transition:
* **Four 2-D spectra**, $20\log_{10}|\mathcal{F}|$: of $w_c$, of $P(w_c)$, of the coarse error,
  and of $P$(coarse error). All four are on one axis in cycles per **fine** pixel, so the
  coarse grid's whole band is the dashed box.
* **1-D power along each axis**: the fine $R^\ell\mathrm{gt}$, $w_c$, $P(w_c)$, and the two
  errors. These are summed over the other axis, so an image and its interpolation agree
  inside the band.
* **Bottom row**: $|H|$ of $P$, which explains the gap between $w_c$ and $P(w_c)$ near
  the coarse Nyquist.

How to read it:
* A coarse error concentrated near the edge of the box, or along the coarse Nyquist
  lines, is the part $P$ mostly removes.
* A coarse error near the centre crosses at full strength.

In [ ]:
fig = lf.fig_correction_spectra(z, meta, outer=OUTER, flip=FLIP)
plt.show()

## 4. Every V-cycle

The correction figure for each outer V-cycle, to see when the coarse recon starts to drift
and whether what crosses drifts with it.

In [ ]:
SHOW_ALL = False
if SHOW_ALL:
    for k in range(len(meta["outers"])):
        fig = lf.fig_correction(z, meta, outer=k, flip=FLIP, panel=1.6)
        plt.show()

## 5. Is the coarse level solving its problem, and is it the right problem?

A coarse level whose NRMSE against $R\,\mathrm{gt}$ rises can mean one of two things:
* it fails to solve its own problem, or
* it solves it, and its problem is the wrong one.

The learned coarse level steps on **its own** problem: its own $A_1, B_1, \lambda_1$,
tilted by $\pi$. The fine problem **restricted** to coarse images (Galerkin) is

$$\min_{x_c}\ \tfrac12\|E P x_c - y\|^2 + \lambda_0 \|A_0 P x_c\|_1 \qquad (\times \tfrac14).$$

From the learned level's own start $u_c = R\,x_{\rm pre}$, at transition `level` of V-cycle
`OUTER`, this runs four fixed coarse solvers for `SOLVE_ITERS` LPDS steps each:

| solver | operators | correction $\pi$ |
|---|---|---|
| learned coarse layer 1, iterated | the coarse level's **first** layer $C_1$ | the net's own |
| Galerkin | fine first layer $F_1$ on $P x_c$, code on the **fine** latent grid | correct nested FAS |
| Galerkin, restricted coeffs | $R_z A_0 P$ / $R B_0 P_z$, code on the coarse latent grid (what a Galerkin-prior coarse level would compute in the net) | correct nested FAS |
| Galerkin, no FAS tilt | as Galerkin | $\pi = 0$: the best coarse image under the fine prior alone |

Each is scored per iteration in two ways:
* the coarse image against $R\,\mathrm{gt}$;
* what it would do to the fine level with $\alpha = 1$: $x_{\rm pre} + P(x_c - u_c)$.

The step sizes are the learned $\tau, \theta$ of the layer the operators come from. The
Galerkin modes use the exact Galerkin data term $R E^H(EPx - y)$, whatever coarse operator
the run was trained with. "The fine problem" means the one defined by the fine level's
**first** layer, which is the layer FAS is built on.

**How to read the table**
* **its OWN residual**: one step of $C_1$ (with the net's $\pi$) at the learned level's start
  and at its end. If the change is not smaller at the end, the learned coarse solve did not
  reduce even its first layer's residual. Its later layers step on other objectives.
* **learned coarse layer 1, iterated**: where the learned coarse problem's solution is. If
  its NRMSE climbs well above `start` while `step` goes to 0, the learned coarse problem
  is solved and its solution is a poor image: the wrong problem.
* **Galerkin**: if it stays at or below `start` and the fine NRMSE (α = 1) improves, a
  consistent coarse problem helps, and a Galerkin prior is worth training.
* **no FAS tilt**: how good a coarse image the fine model gives by itself. If this is poor
  too, the coarse problem is hard in itself (the undersampling), not only inconsistent.
* **final step**: whether the solver converged. If it has not, raise `SOLVE_ITERS`. If it
  grows, the learned step size is unstable when iterated, and the result is divergence,
  not a solution.

In [ ]:
SOLVE_ITERS = 300
SOLVE_LEVELS = (0, 1)          # transitions L1 -> L0 and L2 -> L1
solve = {}
for lev in SOLVE_LEVELS:
    solve[lev] = lf.coarse_solve_test(z, meta, live, outer=OUTER, level=lev,
                                      n_iter=SOLVE_ITERS, every=5)
    lf.coarse_solve_table(solve[lev])
    print()

In [ ]:
for lev in SOLVE_LEVELS:
    fig = lf.fig_coarse_solve(solve[lev], z, meta, flip=FLIP)
    plt.show()

## 6. Is the dual code saturated?

The dual step $z^+ = \mathrm{clip}_\lambda(z + \Delta)$ acts as a sparsity mechanism only while
the increments $\Delta$ are small against $\lambda$. Then a noise coefficient's increments
average out and $z$ stays **inside** the box, which means the coefficient is thresholded to
zero. Once $|\Delta| > |z| + \lambda$, the clip lands on the boundary **whatever** $z$ was. The
code's history is erased, and $Dz \approx \lambda D\,\mathrm{sign}(\Delta)$: a noise coefficient
pushes on $x$ as hard as an edge.

$\Delta$ is what the clip adds to $z$:
* $A\bar x$ under option A, where $\pi_z$ is added after the clip;
* $A\bar x - \pi_z$ under option B (and legacy), where $\pi_z$ goes inside.

This section traces every LPDS sweep of each run in `RUNS_SAT` on the same slice; no weight
is changed. For each sweep it reports:
* **sat**: the fraction of coefficients on the clip boundary ($|z^+| \ge 0.99\lambda$).
* **reset**: the fraction with $|\Delta| > |z| + \lambda$, i.e. saturated whatever $z$ was.
* $|\Delta|/\lambda$: median and 90th percentile.
* $|z|/\lambda$ of the incoming code.
* the fraction with $|\pi_z| > \lambda$ (coarse levels).
* $\lambda$ itself.
* **flip**: of the coefficients saturated now *and* in the previous sweep at the same level,
  the fraction whose phase turned by more than 90 degrees. Each one is a code move of
  about $2\lambda$. In a saturated (phase-code) regime, this is what turns into image noise.
* **push**: $\|	au D(	ext{code change})\| / \|x\|$, how hard that change pushes on $x$.
* **step**: $\|x^+ - x\| / \|x\|$, the whole primal step.

The **anatomy** scope counts only latent positions inside the organ mask (from the section-1
trace), so the zero background doesn't dilute the fractions. Local prox only. Two self-checks
must stay at 0: a clip output outside the box, and a reset coefficient that is not on the
boundary. The table warns if either fails.

How to read it:
* sat and reset are high in every net, the flat one included. That is how LPDS trains here,
  so they don't separate A from B.
* Compare **flip** and **push** across levels and nets. If A's coarse levels flip and push
  much more than B's and the flat net's, that is where A's noise comes from.
* The figure is drawn for the multigrid runs only, since the flat net has only level 0.

In [ ]:
RUNS_SAT = {"A (fixed_point)": "trained_nets/mg_recon/knee/mg81v6vtxA_R8",
            "B (gradient)":    "trained_nets/mg_recon/knee/mg81v6vtxB_R8",
            "flat LPDS":       "trained_nets/mg_recon/knee/lpdsnet_R8"}   # level 0 only
LEVELS_SAT = (0, 1, 2)
OUTERS_SAT = [0, -1]           # first and last V-cycle
SCOPE_SAT  = "anatomy"         # or "all"
sat = {}
for label, run in RUNS_SAT.items():
    sat[label] = lf.trace_saturation(run, VOLUME, SLICE, levels=LEVELS_SAT, outers=OUTERS_SAT,
                                     z=z, sigma=SIGMA, val_kspace_type=VAL_KSPACE)
    lf.saturation_table(*sat[label], scope=SCOPE_SAT, per_sweep=False)
    print()

In [ ]:
fig = lf.fig_saturation({k: v for k, v in sat.items() if k != "flat LPDS"}, scope=SCOPE_SAT)
plt.show()

SHOW_SWEEPS = False            # every sweep, not just the per-level means
if SHOW_SWEEPS:
    for label in sat:
        lf.saturation_table(*sat[label], scope=SCOPE_SAT, per_sweep=True)
        print()

## 7. The coarse-correction steps $\alpha_x$, $\alpha_z$

$$x \leftarrow x + \alpha_x P(w_c - u_c), \qquad z \leftarrow z + \alpha_z P(w^z_c - u^z_c).$$

$\alpha_x$ is a single number (C = 1) and is in wandb as it is. $\alpha_z$ has one value per
dual channel (M of them). That is over the 64-element cutoff of the wandb parameter logger,
so until 2026-10-26 only its gradient norm was logged. New runs log its mean, min and max
under `params/alpha_z/...`.

A collapsed $\alpha_z$ switches off the **dual** correction even when $\alpha_x$ looks
healthy. This section reads both off each checkpoint; it needs no data.

* **Table**: per V-cycle and correction, $\alpha_x$; the mean, min and max of $\alpha_z$;
  the share of channels at or below `NEAR`, i.e. effectively off; and the share that is
  negative.
* **Heatmap**: one row per V-cycle and correction, one column per channel, in the
  channels' own order, so a channel that is off in every V-cycle shows as a white column.
  White means 0; blue is negative, red positive; $\alpha_x$ is beside each row.

Under `alpha_mode="fixed"` both steps are 1 by construction.

In [ ]:
RUNS_ALPHA = ["trained_nets/mg_recon/knee/mg81v6vtxAnest_R8",
              "trained_nets/mg_recon/knee/mg81v6vtxBnest_R8",
              "trained_nets/mg_recon/knee/mg81v6vtxAnestU_R8"]
NEAR = 0.05                       # |alpha| at or below this counts as switched off
alphas = {}
for run in RUNS_ALPHA:
    alphas[os.path.basename(run)] = lf.alpha_records(run)
    lf.alpha_table(alphas[os.path.basename(run)], near=NEAR)
    print()
fig = lf.fig_alpha_z(alphas)
plt.show()